# Random Forest (RF) Model Selection

This notebook evaluates the 10-fold cross-validation parameter grid search for **Random Forest (RF)** across two benchmark datasets:
- **Soybean**
- **Hypothyroid**

### Methodology (Demšar, 2006)
In accordance with sound statistical comparison protocols, candidate configurations are evaluated as complete models rather than isolating each individual parameter.

For each dataset, we conduct two distinct, complementary model selections:
1. **Accuracy-Oriented Selection**: Identifies the configuration maximizing predictive accuracy across folds (tie-breaking with prediction latency).
2. **Efficiency-Oriented Selection**: Identifies the configuration minimizing prediction latency (inference time in milliseconds) while retaining competitive accuracy.

Both selections apply Demšar's methodology:
- Friedman Omnibus test across candidate configurations evaluated over $N=10$ independent folds
- Critical Difference (CD) diagram for statistically equivalent cliques
- Wilcoxon Signed-Rank Test between rank #1 and rank #2


In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Setup paths dynamically
current_dir = Path().resolve()
if (current_dir / "Work2").exists():
    ROOT = current_dir / "Work2"
elif current_dir.name == "analysis":
    ROOT = current_dir.parent
else:
    ROOT = current_dir
sys.path.insert(0, str(ROOT))

from analysis.statistical_utils import (
    load_fold_results,
    load_summary_results,
    run_friedman_analysis,
    select_best_configuration,
    RESULTS_DIR
)
from KNNClassifier import (
    friedman_test,
    wilcoxon_test,
    plot_critical_difference_diagram
)

print(f"Results directory: {RESULTS_DIR}")


Results directory: C:\Users\miche\OneDrive\Desktop\MICHELLE\UPC SPAIN\SEMESTER 1\IML\LABS\IML-Lab-2026\Work2\results


## 1. Soybean Dataset: Random Forest Evaluation


In [2]:
# Dynamically load RF results for Soybean
df_rf_soy_folds = load_fold_results("rf", "soybean")
df_rf_soy_summary = load_summary_results("rf", "soybean")

print(f"Loaded {len(df_rf_soy_folds)} fold evaluations across {len(df_rf_soy_summary)} distinct RF configurations.")


Loaded 360 fold evaluations across 27 distinct RF configurations.


### 1.1 Soybean: Best RF Model Selected by Accuracy


In [3]:
# 1. Best RF Configuration Selected by Accuracy
print("Top 10 RF Configurations by Accuracy (Soybean):")
display(df_rf_soy_summary.sort_values(by=["accuracy", "prediction_time"], ascending=[False, True])[
    ["model", "n_estimators", "criterion", "max_depth", "accuracy", "accuracy_std", "prediction_time_ms"]
].head(10))

res_rf_soy_acc = select_best_configuration(
    df_rf_soy_folds, df_rf_soy_summary, dataset="soybean", algo_name="RF", metric="accuracy"
)

print("\n=== Selected Best RF Model by Accuracy (Soybean) ===")
print(f"  Model:                 {res_rf_soy_acc['best_model']}")
print(f"  Accuracy:              {res_rf_soy_acc['meta']['mean_accuracy']*100:.2f}% ± {res_rf_soy_acc['meta']['std_accuracy']*100:.2f}%")
print(f"  Prediction Time:       {res_rf_soy_acc['meta']['mean_prediction_time_ms']:.2f} ms")
print(f"  Friedman Chi2 (acc):   {res_rf_soy_acc['meta']['friedman_chi2']:.4f} (p-value: {res_rf_soy_acc['meta']['friedman_p']:.4e})")
print(f"  Wilcoxon #1 vs #2 (p): {res_rf_soy_acc['meta']['wilcoxon_best_vs_second_p']:.4f}")


Top 10 RF Configurations by Accuracy (Soybean):


,model,n_estimators,criterion,max_depth,accuracy,accuracy_std,prediction_time_ms
1,"RF(n=200, crit=log_loss, depth=10)",200,log_loss,10,0.940491,0.024176,42.74482
0,"RF(n=200, crit=entropy, depth=10)",200,entropy,10,0.940491,0.024176,42.96465
2,"RF(n=100, crit=entropy, depth=10)",100,entropy,10,0.938961,0.021079,21.94947
3,"RF(n=100, crit=log_loss, depth=10)",100,log_loss,10,0.938961,0.021079,22.08056
4,"RF(n=200, crit=gini, depth=10)",200,gini,10,0.938590,0.022807,43.22994
6,"RF(n=200, crit=log_loss, depth=20)",200,log_loss,20,0.937585,0.022373,43.80267
5,"RF(n=200, crit=entropy, depth=20)",200,entropy,20,0.937585,0.022373,44.26968
7,"RF(n=50, crit=entropy, depth=10)",50,entropy,10,0.935910,0.023828,11.72979
8,"RF(n=50, crit=log_loss, depth=10)",50,log_loss,10,0.935910,0.023828,11.91419
9,"RF(n=200, crit=gini, depth=20)",200,gini,20,0.934508,0.021469,43.99078



=== Selected Best RF Model by Accuracy (Soybean) ===
  Model:                 RF(n=200, crit=log_loss, depth=10)
  Accuracy:              94.05% ± 2.42%
  Prediction Time:       42.74 ms
  Friedman Chi2 (acc):   182.0421 (p-value: 2.2910e-25)
  Wilcoxon #1 vs #2 (p): 1.0000


### 1.2 Soybean: Best RF Model Selected by Efficiency (Prediction Latency)


In [4]:
# 2. Best RF Configuration Selected by Efficiency (Lowest Prediction Time)
print("Top 10 RF Configurations by Efficiency / Prediction Time (Soybean):")
display(df_rf_soy_summary.sort_values(by=["prediction_time", "accuracy"], ascending=[True, False])[
    ["model", "n_estimators", "criterion", "max_depth", "accuracy", "accuracy_std", "prediction_time_ms"]
].head(10))

res_rf_soy_eff = select_best_configuration(
    df_rf_soy_folds, df_rf_soy_summary, dataset="soybean", algo_name="RF", metric="prediction_time"
)

print("\n=== Selected Best RF Model by Efficiency (Soybean) ===")
print(f"  Model:                 {res_rf_soy_eff['best_model']}")
print(f"  Accuracy:              {res_rf_soy_eff['meta']['mean_accuracy']*100:.2f}% ± {res_rf_soy_eff['meta']['std_accuracy']*100:.2f}%")
print(f"  Prediction Time:       {res_rf_soy_eff['meta']['mean_prediction_time_ms']:.2f} ms")
print(f"  Friedman Chi2 (time):  {res_rf_soy_eff['meta']['friedman_chi2']:.4f} (p-value: {res_rf_soy_eff['meta']['friedman_p']:.4e})")
print(f"  Wilcoxon #1 vs #2 (p): {res_rf_soy_eff['meta']['wilcoxon_best_vs_second_p']:.4f}")


Top 10 RF Configurations by Efficiency / Prediction Time (Soybean):


,model,n_estimators,criterion,max_depth,accuracy,accuracy_std,prediction_time_ms
23,"RF(n=50, crit=gini, depth=5)",50,gini,5,0.877323,0.038245,11.58718
26,"RF(n=50, crit=log_loss, depth=5)",50,log_loss,5,0.867571,0.044233,11.61355
7,"RF(n=50, crit=entropy, depth=10)",50,entropy,10,0.935910,0.023828,11.72979
15,"RF(n=50, crit=log_loss, depth=20)",50,log_loss,20,0.933048,0.021408,11.75458
25,"RF(n=50, crit=entropy, depth=5)",50,entropy,5,0.867571,0.044233,11.75689
17,"RF(n=50, crit=gini, depth=10)",50,gini,10,0.932774,0.027499,11.79564
14,"RF(n=50, crit=entropy, depth=20)",50,entropy,20,0.933048,0.021408,11.87280
8,"RF(n=50, crit=log_loss, depth=10)",50,log_loss,10,0.935910,0.023828,11.91419
12,"RF(n=50, crit=gini, depth=20)",50,gini,20,0.934252,0.025355,12.11703
21,"RF(n=100, crit=log_loss, depth=5)",100,log_loss,5,0.880617,0.038409,21.41521



=== Selected Best RF Model by Efficiency (Soybean) ===
  Model:                 RF(n=50, crit=gini, depth=5)
  Accuracy:              87.73% ± 3.82%
  Prediction Time:       11.59 ms
  Friedman Chi2 (time):  239.7048 (p-value: 1.8096e-36)
  Wilcoxon #1 vs #2 (p): 0.9219


## 2. Hypothyroid Dataset: Random Forest Evaluation


In [5]:
# Dynamically load RF results for Hypothyroid
df_rf_hypo_folds = load_fold_results("rf", "hypothyroid")
df_rf_hypo_summary = load_summary_results("rf", "hypothyroid")

print(f"Loaded {len(df_rf_hypo_folds)} fold evaluations across {len(df_rf_hypo_summary)} distinct RF configurations.")


Loaded 360 fold evaluations across 24 distinct RF configurations.


### 2.1 Hypothyroid: Best RF Model Selected by Accuracy


In [6]:
# 1. Best RF Configuration Selected by Accuracy
print("Top 10 RF Configurations by Accuracy (Hypothyroid):")
display(df_rf_hypo_summary.sort_values(by=["accuracy", "prediction_time"], ascending=[False, True])[
    ["model", "n_estimators", "criterion", "max_depth", "accuracy", "accuracy_std", "prediction_time_ms"]
].head(10))

res_rf_hypo_acc = select_best_configuration(
    df_rf_hypo_folds, df_rf_hypo_summary, dataset="hypothyroid", algo_name="RF", metric="accuracy"
)

print("\n=== Selected Best RF Model by Accuracy (Hypothyroid) ===")
print(f"  Model:                 {res_rf_hypo_acc['best_model']}")
print(f"  Accuracy:              {res_rf_hypo_acc['meta']['mean_accuracy']*100:.2f}% ± {res_rf_hypo_acc['meta']['std_accuracy']*100:.2f}%")
print(f"  Prediction Time:       {res_rf_hypo_acc['meta']['mean_prediction_time_ms']:.2f} ms")
print(f"  Friedman Chi2 (acc):   {res_rf_hypo_acc['meta']['friedman_chi2']:.4f} (p-value: {res_rf_hypo_acc['meta']['friedman_p']:.4e})")
print(f"  Wilcoxon #1 vs #2 (p): {res_rf_hypo_acc['meta']['wilcoxon_best_vs_second_p']:.4f}")


Top 10 RF Configurations by Accuracy (Hypothyroid):


,model,n_estimators,criterion,max_depth,accuracy,accuracy_std,prediction_time_ms
1,"RF(n=200, crit=log_loss, depth=10)",200,log_loss,10,0.995762,0.003106,12.90982
0,"RF(n=200, crit=entropy, depth=10)",200,entropy,10,0.995762,0.003106,12.96480
3,"RF(n=100, crit=log_loss, depth=10)",100,log_loss,10,0.995497,0.002803,6.96271
2,"RF(n=100, crit=entropy, depth=10)",100,entropy,10,0.995497,0.002803,7.09994
4,"RF(n=200, crit=gini, depth=10)",200,gini,10,0.994969,0.004572,12.92036
6,"RF(n=50, crit=log_loss, depth=10)",50,log_loss,10,0.994702,0.003302,4.06089
5,"RF(n=50, crit=entropy, depth=10)",50,entropy,10,0.994702,0.003302,4.10192
7,"RF(n=100, crit=gini, depth=10)",100,gini,10,0.994176,0.006208,7.06470
8,"RF(n=50, crit=gini, depth=10)",50,gini,10,0.992849,0.003747,4.04205
9,"RF(n=10, crit=entropy, depth=10)",10,entropy,10,0.989915,0.010173,1.68218



=== Selected Best RF Model by Accuracy (Hypothyroid) ===
  Model:                 RF(n=200, crit=log_loss, depth=10)
  Accuracy:              99.58% ± 0.31%
  Prediction Time:       12.91 ms
  Friedman Chi2 (acc):   197.4121 (p-value: 1.1114e-29)
  Wilcoxon #1 vs #2 (p): 1.0000


### 2.2 Hypothyroid: Best RF Model Selected by Efficiency (Prediction Latency)


In [7]:
# 2. Best RF Configuration Selected by Efficiency (Lowest Prediction Time)
print("Top 10 RF Configurations by Efficiency / Prediction Time (Hypothyroid):")
display(df_rf_hypo_summary.sort_values(by=["prediction_time", "accuracy"], ascending=[True, False])[
    ["model", "n_estimators", "criterion", "max_depth", "accuracy", "accuracy_std", "prediction_time_ms"]
].head(10))

res_rf_hypo_eff = select_best_configuration(
    df_rf_hypo_folds, df_rf_hypo_summary, dataset="hypothyroid", algo_name="RF", metric="prediction_time"
)

print("\n=== Selected Best RF Model by Efficiency (Hypothyroid) ===")
print(f"  Model:                 {res_rf_hypo_eff['best_model']}")
print(f"  Accuracy:              {res_rf_hypo_eff['meta']['mean_accuracy']*100:.2f}% ± {res_rf_hypo_eff['meta']['std_accuracy']*100:.2f}%")
print(f"  Prediction Time:       {res_rf_hypo_eff['meta']['mean_prediction_time_ms']:.2f} ms")
print(f"  Friedman Chi2 (time):  {res_rf_hypo_eff['meta']['friedman_chi2']:.4f} (p-value: {res_rf_hypo_eff['meta']['friedman_p']:.4e})")
print(f"  Wilcoxon #1 vs #2 (p): {res_rf_hypo_eff['meta']['wilcoxon_best_vs_second_p']:.4f}")


Top 10 RF Configurations by Efficiency / Prediction Time (Hypothyroid):


,model,n_estimators,criterion,max_depth,accuracy,accuracy_std,prediction_time_ms
14,"RF(n=10, crit=gini, depth=5)",10,gini,5,0.973214,0.018363,1.60337
12,"RF(n=10, crit=log_loss, depth=5)",10,log_loss,5,0.976394,0.015282,1.63366
13,"RF(n=10, crit=entropy, depth=5)",10,entropy,5,0.976394,0.015282,1.64776
9,"RF(n=10, crit=entropy, depth=10)",10,entropy,10,0.989915,0.010173,1.68218
11,"RF(n=10, crit=gini, depth=10)",10,gini,10,0.989136,0.006166,1.70231
10,"RF(n=10, crit=log_loss, depth=10)",10,log_loss,10,0.989915,0.010173,1.72216
21,"RF(n=50, crit=entropy, depth=5)",50,entropy,5,0.949365,0.006158,3.82385
15,"RF(n=50, crit=gini, depth=5)",50,gini,5,0.954661,0.017200,3.89341
22,"RF(n=50, crit=log_loss, depth=5)",50,log_loss,5,0.949365,0.006158,3.96150
8,"RF(n=50, crit=gini, depth=10)",50,gini,10,0.992849,0.003747,4.04205



=== Selected Best RF Model by Efficiency (Hypothyroid) ===
  Model:                 RF(n=10, crit=gini, depth=5)
  Accuracy:              97.32% ± 1.84%
  Prediction Time:       1.60 ms
  Friedman Chi2 (time):  221.3760 (p-value: 2.2866e-34)
  Wilcoxon #1 vs #2 (p): 0.3750


## 3. Dynamically Generated Summary: Best Random Forest Models


In [8]:
rf_summary_rows = [
    {
        "Dataset": "Soybean",
        "Criterion": "Best Accuracy",
        "Best Configuration": res_rf_soy_acc["best_model"],
        "Accuracy (%)": f"{res_rf_soy_acc['meta']['mean_accuracy']*100:.2f}% ± {res_rf_soy_acc['meta']['std_accuracy']*100:.2f}%",
        "Pred Time (ms)": f"{res_rf_soy_acc['meta']['mean_prediction_time_ms']:.2f}",
    },
    {
        "Dataset": "Soybean",
        "Criterion": "Best Efficiency",
        "Best Configuration": res_rf_soy_eff["best_model"],
        "Accuracy (%)": f"{res_rf_soy_eff['meta']['mean_accuracy']*100:.2f}% ± {res_rf_soy_eff['meta']['std_accuracy']*100:.2f}%",
        "Pred Time (ms)": f"{res_rf_soy_eff['meta']['mean_prediction_time_ms']:.2f}",
    },
    {
        "Dataset": "Hypothyroid",
        "Criterion": "Best Accuracy",
        "Best Configuration": res_rf_hypo_acc["best_model"],
        "Accuracy (%)": f"{res_rf_hypo_acc['meta']['mean_accuracy']*100:.2f}% ± {res_rf_hypo_acc['meta']['std_accuracy']*100:.2f}%",
        "Pred Time (ms)": f"{res_rf_hypo_acc['meta']['mean_prediction_time_ms']:.2f}",
    },
    {
        "Dataset": "Hypothyroid",
        "Criterion": "Best Efficiency",
        "Best Configuration": res_rf_hypo_eff["best_model"],
        "Accuracy (%)": f"{res_rf_hypo_eff['meta']['mean_accuracy']*100:.2f}% ± {res_rf_hypo_eff['meta']['std_accuracy']*100:.2f}%",
        "Pred Time (ms)": f"{res_rf_hypo_eff['meta']['mean_prediction_time_ms']:.2f}",
    },
]

df_best_rf_summary = pd.DataFrame(rf_summary_rows)
print("=== Final Selected Best Random Forest Models Summary (Accuracy vs Efficiency) ===")
display(df_best_rf_summary)


=== Final Selected Best Random Forest Models Summary (Accuracy vs Efficiency) ===


,Dataset,Criterion,Best Configuration,Accuracy (%),Pred Time (ms)
0,Soybean,Best Accuracy,"RF(n=200, crit=log_loss, depth=10)",94.05% ± 2.42%,42.74
1,Soybean,Best Efficiency,"RF(n=50, crit=gini, depth=5)",87.73% ± 3.82%,11.59
2,Hypothyroid,Best Accuracy,"RF(n=200, crit=log_loss, depth=10)",99.58% ± 0.31%,12.91
3,Hypothyroid,Best Efficiency,"RF(n=10, crit=gini, depth=5)",97.32% ± 1.84%,1.60


## 4. Key Takeaways and Discussion

1. **Ensemble Size and Tree Depth Dynamics**:
   - On **Soybean**, larger ensembles (`n=200`) with deeper trees (`depth=10`) maximize accuracy (94.05%), but demand 42.74 ms prediction time. Conversely, a shallower ensemble (`n=50, depth=5`) drops prediction time to 11.59 ms (~3.7x speedup).
   - On **Hypothyroid**, `n=200, depth=10` reaches near-perfect accuracy (99.58%, 12.91 ms). A minimal ensemble (`n=10, depth=5`) achieves 97.32% accuracy while requiring only 1.60 ms (~8.1x speedup).
2. **Computational Scaling**:
   - Inference latency scales strictly linearly with `n_estimators` and the average depth traversed, offering an explicit knob for latency-critical applications.
